# Monte Carlo vs Black-Scholes: Pricing European Options

This notebook prices European calls and puts by Monte Carlo simulation and compares the estimates with the analytical Black-Scholes prices. All pricing functions are imported from `European_Option.py`, so the notebook and the script always use the same code.

## Theory

### Geometric Brownian motion
The underlying asset is assumed to follow geometric Brownian motion (GBM):

$$\frac{dS_t}{S_t} = \mu\,dt + \sigma\,dW_t$$

where $\mu$ is the drift, $\sigma$ the volatility and $W_t$ a standard Brownian motion. Applying Itô's lemma to $\ln S_t$ gives the exact solution

$$S_t = S_0 \exp\left(\left(\mu - \tfrac{1}{2}\sigma^2\right)t + \sigma W_t\right).$$

The $-\tfrac{1}{2}\sigma^2$ term is the **Itô correction**. Without it, the simulated prices grow too fast on average: $\mathbb{E}[S_t]$ would be $S_0 e^{(\mu + \sigma^2/2)t}$ instead of $S_0 e^{\mu t}$.

### Risk-neutral pricing
The fair price of an option is its expected discounted payoff under the **risk-neutral measure** $\mathbb{Q}$, where every asset grows at the risk-free rate $r$. So for pricing, the real-world drift $\mu$ is replaced by $r$:

$$V_0 = e^{-rT}\,\mathbb{E}^{\mathbb{Q}}\left[\text{payoff}(S_T)\right], \qquad S_T = S_0 \exp\left(\left(r - \tfrac{1}{2}\sigma^2\right)T + \sigma\sqrt{T}\,Z\right),\quad Z \sim N(0,1).$$

The real-world drift $\mu$ has no effect on the option price; here it is used only to illustrate asset paths. A European payoff depends only on $S_T$, which can be sampled exactly in one step, so no time-stepping is needed for pricing.

### Black-Scholes formula
For a call with payoff $\max(S_T - K, 0)$ and a put with payoff $\max(K - S_T, 0)$, the expectation above has a closed form:

$$C = S_0\,N(d_1) - K e^{-rT} N(d_2), \qquad P = K e^{-rT} N(-d_2) - S_0\,N(-d_1),$$

$$d_1 = \frac{\ln(S_0/K) + \left(r + \tfrac{1}{2}\sigma^2\right)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T},$$

where $N$ is the standard normal CDF. Calls and puts are linked by **put-call parity**: $C - P = S_0 - K e^{-rT}$.

### Monte Carlo estimator
With $n$ independent draws of $S_T$, the Monte Carlo price is the sample mean of the discounted payoffs $Y_i = e^{-rT}\,\text{payoff}(S_T^{(i)})$:

$$\hat V = \frac{1}{n}\sum_{i=1}^{n} Y_i, \qquad \text{SE}(\hat V) = \frac{s_Y}{\sqrt{n}},$$

where $s_Y$ is the sample standard deviation of the $Y_i$. By the central limit theorem, $\hat V \pm 1.96\,\text{SE}$ is an approximate 95% confidence interval. The error shrinks like $1/\sqrt{n}$, so each extra decimal place of accuracy costs 100 times more paths.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from European_Option import (
    simulate_paths,
    mc_price,
    black_scholes_price,
)

# Chart style: light surface, recessive hairline grid, ink-coloured text
SURFACE = "#fcfcfb"
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"
BLUE, ORANGE = "#2a78d6", "#eb6834"  # categorical slots 1 and 2

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica Neue", "Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 10,
    "text.color": INK,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.titlecolor": INK,
    "axes.labelcolor": INK_2,
    "axes.edgecolor": AXIS,
    "axes.linewidth": 1,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": GRID,
    "grid.linewidth": 1,
    "grid.linestyle": "-",
    "axes.axisbelow": True,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "xtick.labelcolor": INK_2,
    "ytick.labelcolor": INK_2,
    "lines.linewidth": 2,
    "lines.solid_capstyle": "round",
    "lines.solid_joinstyle": "round",
    "legend.frameon": False,
    "legend.labelcolor": INK_2,
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
})

pd.options.display.float_format = "{:,.4f}".format

## Parameters
The option is out of the money: the strike is 10% above the current price. The seed makes every run reproducible.

In [2]:
S0 = 100          # initial asset price
K = 110           # strike
r = 0.03          # risk-free rate
sigma = 0.3       # volatility
T = 1.0           # time to expiry (years)
n_paths = 100_000 # Monte Carlo paths
seed = 42

mu = 0.10         # real-world drift, used only for the path illustration